# Hierarchical Bayesian inference of the BBH merger time-delay scale

Infer the exponential delay timescale $\tau$ using gravitational-wave posterior samples and a selection function computed with GWSNR. We fit only the shape of the intrinsic redshift distribution, not the total merger rate. Cosmology and all other population parameters are fixed.

This notebook follows the selection-corrected likelihood used in GW population studies. The physical redshift model from the SFR and time-delay notebook will be implemented separately. A clearly labelled toy model is provided for checking the likelihood, not for scientific inference.

## 1. Population model

Let $\theta=(z_s,\lambda)$, where $\lambda$ contains masses, spins and extrinsic parameters. The population model is

$$p(\theta\mid\tau)=p(z_s\mid\tau)p(\lambda\mid z_s).$$

The conditional distribution of the remaining parameters is fixed, but may depend on redshift. From the previous notebook,

$$p(z_s\mid\tau)=\frac{R_m(z_s\mid\tau)(dV_c/dz_s)/(1+z_s)}{\int_0^{z_{\max}}R_m(z\mid\tau)(dV_c/dz)/(1+z)\,dz}.$$

This is the redshift PDF of all mergers per unit observer time, before detection. The local rate normalization cancels. The factor $1/(1+z_s)$ must be included once only. The PDF is normalized over the adopted support, which must cover the intended population and injection proposal.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import trapezoid
import bilby

# The redshift grid used by the population model
z_max = 10
z_grid = np.linspace(0, z_max, 2001)

# Physical model to be implemented later
# Return a normalized PDF evaluated on z_grid.
def redshift_distribution(tau):
    raise NotImplementedError("Connect the SFR + time-delay model here.")

## 2. Reuse individual-event posterior samples

For event $i$, let $\pi_{0,i}(\theta)$ be its PE prior and $\mathcal{Z}_{0,i}$ its evidence. Bayes' theorem gives

$$p_0(\theta\mid d_i)=\frac{p(d_i\mid\theta)\pi_{0,i}(\theta)}{\mathcal{Z}_{0,i}}.$$

For $n_i$ equally weighted posterior samples, the event likelihood, divided by its PE evidence, is estimated by

$$I_i(\tau)=\frac{1}{n_i}\sum_{k=1}^{n_i}p(z_{ik}\mid\tau)\frac{p(\lambda_{ik}\mid z_{ik})}{\pi_{0,i}(\theta_{ik})}.$$

Define the fixed weight $w_{ik}=p(\lambda_{ik}\mid z_{ik})/\pi_{0,i}(\theta_{ik})$. Compute it only once. If and only if the nuisance-parameter conditional prior is the same as the target population, this reduces to $w_{ik}=1/\pi_{0,i}(z_{ik})$. Not fitting masses or spins does not by itself justify this cancellation.

Use the same coordinates and cosmology in every density. For example, if the PE prior is expressed in $(m_{1,d},m_{2,d},d_L)$ but the analysis uses $(m_{1,s},m_{2,s},z_s)$, the joint density transforms with the factor $(1+z_s)^2|dd_L/dz_s|$. For a distance-only prior, use $\pi_0(z_s)=\pi_0(d_L(z_s))|dd_L/dz_s|$. For other mass coordinates the Jacobian differs. Do not multiply by a Jacobian again if the supplied prior is already in the desired coordinates.

**Input format:** one DataFrame per event, with columns `redshift`, `prior` (joint PE density) and `p_other` (fixed conditional population density). Alternatively, use the simpler format below only when cancellation holds: `redshift`, `prior_z`. Preserve all posterior samples (including samples outside population support, which contribute zero). Use equally weighted PE samples or resample weighted samples correctly. The PE prior must cover the target population.

In [ ]:
# Prepare the event weights once, before sampling.
def prepare_events(posteriors):
    events = []
    for posterior in posteriors:
        z = posterior["redshift"].to_numpy()
        weight = (posterior["p_other"] / posterior["prior"]).to_numpy()
        events.append((z, weight))
    return events

# Alternative: only if other conditional priors cancel.
def prepare_redshift_only_events(posteriors):
    events = []
    for posterior in posteriors:
        z = posterior["redshift"].to_numpy()
        weight = 1 / posterior["prior_z"].to_numpy()
        events.append((z, weight))
    return events

## 3. Selection effects using GWSNR

The probability that a random astrophysical merger enters the catalog is

$$\alpha(\tau)=\int P_{\rm det}(z,\lambda)p(z\mid\tau)p(\lambda\mid z)\,dz\,d\lambda.$$

Generate a fixed set of injections from a known proposal $q(z,\lambda)$. Estimate the integral by importance sampling:

$$\widehat\alpha(\tau)=\frac{1}{M}\sum_{j=1}^{M}p(z_j\mid\tau)\frac{P_{{\rm det},j}p(\lambda_j\mid z_j)}{q(z_j,\lambda_j)}.$$

$M$ is the number of **all generated injections**, not the number detected. If the injections use $q(z,\lambda)=q_z(z)p(\lambda\mid z)$, the fixed weight becomes $P_{{\rm det},j}/q_z(z_j)$. Choose a broad proposal with support over the target population; use the actual proposal density, not a detected-redshift histogram.

Compute GWSNR probabilities before running Bilby, for example using the documented GWSNR API (adjust PSD, detector network, thresholds and waveform to the survey):

```python
from gwsnr import GWSNR
calculator = GWSNR()  # configure for the observing run
snrs = calculator.optimal_snr(**gw_parameters)
pdet = calculator.pdet(snr_dict=snrs, snr_th=8,
                       snr_th_net=8, type="matched_filter")["pdet_net"]
```

Here `gw_parameters` is a dictionary of injection arrays in GWSNR conventions, including detector-frame masses $m_d=(1+z)m_s$, luminosity distance in Mpc, spins, orientation, sky position and GPS time. The SNR threshold of 8 is illustrative, not an LVK FAR cut. Real catalog analyses require validating or calibrating GWSNR probabilities against search injections at the same FAR, detector sensitivity, duty cycle and catalog cuts. If using actual found/missed injections, replace $P_{\rm det}$ by 1/0. If only found injections are retained, divide their weight sum by the total generated count, not the length of the found subset.

For several observing periods, either use one effective survey with $P_{\rm det}=\sum_r T_r P_{{\rm det},r}/\sum_r T_r$ (conditioning on total $N$, common astrophysical rate) or condition separately on counts $N_r$ and use $-\sum_r N_r\log\alpha_r$ in the likelihood. Do not use just O4 sensitivity for a cumulative catalog without modelling earlier runs.

**Input format:** a DataFrame with `redshift`, `pdet`, `p_draw` (joint injection density) and `p_other`. For matched conditional draws the alternative uses `redshift`, `pdet`, `q_z`. All rows, including missed injections, must be retained for the code below.

In [ ]:
def prepare_injections(injections):
    z = injections["redshift"].to_numpy()
    weight = (injections["pdet"] * injections["p_other"]
              / injections["p_draw"]).to_numpy()
    return z, weight

# Alternative: injections drawn from the fixed nuisance population.
def prepare_redshift_only_injections(injections):
    z = injections["redshift"].to_numpy()
    weight = (injections["pdet"] / injections["q_z"]).to_numpy()
    return z, weight

## 4. Hierarchical likelihood

Conditioning on the number $N$ of detections, and assuming independent astrophysical events,

$$\mathcal L(\tau)\propto\alpha(\tau)^{-N}\prod_{i=1}^N I_i(\tau),$$

$$\log\mathcal L(\tau)=\sum_i\log I_i(\tau)-N\log\alpha(\tau).$$

The per-event PE evidences are independent of $\tau$ and can be dropped for inference of $\tau$. We do not include $P_{\rm det}$ in the event numerator: detection is a condition on the observed data, and the efficiency belongs in the catalog denominator. An alternative Poisson likelihood is needed if fitting the absolute merger rate as well. The normalized redshift distribution here is evaluated once per likelihood call, then interpolated to the event and injection redshifts.

The class below uses Bilby's parameter-dictionary interface (Bilby 2.7 or newer). It is kept short; the input preparation is independent of the sampler.

In [ ]:
def population_pdf(z, pdf_grid):
    # No population probability outside the adopted redshift support.
    return np.interp(z, z_grid, pdf_grid, left=0, right=0)


def selection_function(pdf_grid, injections):
    z, weight = injections
    return np.mean(population_pdf(z, pdf_grid) * weight)


def catalog_log_likelihood(pdf_grid, events, injections):
    alpha = selection_function(pdf_grid, injections)
    if not np.isfinite(alpha) or alpha <= 0:
        return -np.inf

    log_likelihood = -len(events) * np.log(alpha)

    for z, weight in events:
        event_integral = np.mean(population_pdf(z, pdf_grid) * weight)
        if not np.isfinite(event_integral) or event_integral <= 0:
            return -np.inf
        log_likelihood += np.log(event_integral)

    return log_likelihood


class RedshiftLikelihood(bilby.Likelihood):
    def __init__(self, events, injections, redshift_model):
        super().__init__()
        self.events = events
        self.injections = injections
        self.redshift_model = redshift_model

    def log_likelihood(self, parameters):
        tau = parameters["tau"]
        pdf_grid = self.redshift_model(tau)
        return catalog_log_likelihood(pdf_grid, self.events, self.injections)

## 5. Check the Monte Carlo accuracy

For a set of importance weights $a_j$, the effective sample size is $n_{\rm eff}=(\sum_j a_j)^2/\sum_j a_j^2$. Check this for each event and for the injection sum over the hyperprior and posterior. The uncertainty in selection can be amplified by $N$ in the log-likelihood. More injections and/or a better proposal may be required. For independent draws, an approximate pointwise variance of the log likelihood is the sum of event relative-variance estimates plus $N^2$ times the selection relative variance. These are diagnostics, not a full treatment of Monte Carlo uncertainty or GWSNR modelling systematics. See Farr (2019) and GWTC-4 Appendix A. Check convergence by increasing the number of posterior and injection samples and refining the redshift grid.

In [ ]:
def effective_sample_size(weights):
    if np.sum(weights**2) == 0:
        return 0
    return np.sum(weights)**2 / np.sum(weights**2)


def check_weights(pdf_grid, events, injections):
    z, weight = injections
    selection_weights = population_pdf(z, pdf_grid) * weight
    print("Selection efficiency:", np.mean(selection_weights))
    print("Injection ESS:", effective_sample_size(selection_weights))
    event_ess = []
    for z, weight in events:
        event_ess.append(effective_sample_size(population_pdf(z, pdf_grid) * weight))
    print("Minimum event ESS:", min(event_ess))

## 6. Toy test (not the physical SFR model)

This test uses a truncated exponential PDF in redshift with a dimensionless toy shape parameter, a known selection curve, and two artificial posterior samples. It verifies the interface without implementing the astrophysical delay model. The toy parameter is NOT a physical time delay in Gyr. All posterior and injection samples here have a uniform redshift prior/proposal. The data are artificial; no scientific inference should be drawn from them.

In [ ]:
def toy_redshift_distribution(tau):
    pdf = np.exp(-z_grid / tau)
    return pdf / trapezoid(pdf, z_grid)

rng = np.random.default_rng(123)

# Artificial posterior samples (not actual GW events)
toy_posteriors = []
for center in [0.4, 0.8]:
    z_samples = rng.normal(center, 0.12, 3000)
    z_samples = z_samples[(z_samples >= 0) & (z_samples <= z_max)]
    toy_posteriors.append(pd.DataFrame({
        "redshift": z_samples,
        "prior_z": 1 / z_max
    }))

# Draw all injections from a uniform proposal in redshift.
z_inj = rng.uniform(0, z_max, 100000)
toy_injections = pd.DataFrame({
    "redshift": z_inj,
    "q_z": 1 / z_max,
    "pdet": np.exp(-z_inj)
})

toy_events = prepare_redshift_only_events(toy_posteriors)
toy_selection = prepare_redshift_only_injections(toy_injections)
pdf_grid = toy_redshift_distribution(1)

# Check normalization and selection against direct quadrature.
print("PDF integral:", trapezoid(pdf_grid, z_grid))
print("Selection (quadrature):", trapezoid(pdf_grid * np.exp(-z_grid), z_grid))
check_weights(pdf_grid, toy_events, toy_selection)

# Complete detection should give alpha approximately 1.
all_detected = toy_injections.copy()
all_detected["pdet"] = 1
print("Complete detection:", selection_function(
    pdf_grid, prepare_redshift_only_injections(all_detected)))

likelihood = RedshiftLikelihood(toy_events, toy_selection,
                                toy_redshift_distribution)
print("Toy log likelihood:", likelihood.log_likelihood({"tau": 1}))

## 7. Run Bilby

For the physical model, a log-uniform hyperprior between 0.1 and 10 Gyr is an illustrative choice; test sensitivity to this choice. Run the following example with the toy likelihood first. The toy prior uses the same numeric bounds for demonstration only. A one-dimensional grid is also useful to cross-check nested sampling.

In [ ]:
priors = dict(tau=bilby.core.prior.LogUniform(0.1, 10, name="tau"))

# Uncomment to run the toy inference.
# result = bilby.run_sampler(
#     likelihood=likelihood, priors=priors,
#     sampler="dynesty", nlive=500,
#     outdir="outdir", label="toy_redshift", resume=False
# )
# result.plot_corner()

## 8. Connect real data and the physical redshift model

After preparing actual event posteriors and injections (with their matching priors and population densities), replace the toy inputs. The efficient implementation of `redshift_distribution(tau)` is intentionally left for the next notebook. It must return a nonnegative PDF normalized on `z_grid` for each positive $\tau$.

The following is a template, not executable until the actual files and physical model are supplied. Each posterior CSV must contain a joint PE prior density and the fixed nuisance population density. For real LVK posteriors, reconstruct the event-specific PE priors and coordinate transformations from the release metadata; do not assume a universal uniform-in-comoving-volume prior. Use the matched-conditional alternatives only when their assumptions are satisfied.

In [ ]:
# posterior_files = ["event_1.csv", "event_2.csv"]
# posteriors = [pd.read_csv(file) for file in posterior_files]
# injections = pd.read_csv("injections_with_pdet.csv")
# events = prepare_events(posteriors)
# selection = prepare_injections(injections)
# likelihood = RedshiftLikelihood(events, selection, redshift_distribution)
# result = bilby.run_sampler(
#     likelihood=likelihood, priors=priors,
#     sampler="dynesty", nlive=500,
#     outdir="outdir", label="bbh_delay", resume=False
# )
# result.plot_corner()

## References

- [Vitale et al. (2019), Measuring the star formation rate with gravitational waves from binary black holes](https://arxiv.org/abs/1808.00901).
- [Mandel, Farr & Gair (2019), Extracting distribution parameters from multiple uncertain observations with selection biases](https://arxiv.org/abs/1809.02063).
- [Thrane & Talbot (2019), An introduction to Bayesian inference in gravitational-wave astronomy](https://arxiv.org/abs/1809.02293).
- [Farr (2019), Accuracy Requirements for Empirically-Measured Selection Functions](https://arxiv.org/abs/1904.10879).
- [LVK, GWTC-4.0: Population Properties of Merging Compact Binaries](https://dcc.ligo.org/LIGO-P2400004/public), Appendix A and Section 6.4.
- [LVK, GWTC-5.0: Population Properties of Merging Compact Binaries](https://arxiv.org/abs/2605.27226).
- [GWPopulation hierarchical likelihood](https://colmtalbot.github.io/gwpopulation/autoapi/gwpopulation/hyperpe/index.html).
- [Bilby likelihood interface](https://bilby-dev.github.io/bilby/likelihood.html).
- [GWSNR detection probability](https://gwsnr.hemantaph.com/probabilityofdetection.html).